# CS3807 – Deep Learning Laboratory
## Experiment 5 – Modified Part 1
**Comprehensive Study of CNN Training, Regularization, Optimization, Hyperparameter Tuning, Transfer Learning and Fine-Tuning**

This version is a modified, self-contained implementation based on the supplied Experiment 5 specification. It uses MobileNetV2 and Oxford-IIIT Pet, while changing the seed, batch size, classifier width, learning-rate candidates, dropout setting and fine-tuning depth. Results are generated at runtime; no numerical result is hard-coded.

In [ ]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
NUM_CLASSES = 37
EPOCHS = 3
BATCH_SIZE = 24
OUTPUT_DIR = os.path.join(os.getcwd(), "Ex5_Modified_Outputs")
FIG_DIR = os.path.join(OUTPUT_DIR, "Figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("Seed:", SEED)
print("TensorFlow:", tf.__version__)
print("Output directory:", OUTPUT_DIR)

In [ ]:
plt.figure(figsize=(10, 7))
for i, (image, label) in enumerate(train_raw.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(ds_info.features["label"].names[int(label.numpy())])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
(train_raw, val_raw, test_raw), ds_info = tfds.load(
    "oxford_iiit_pet",
    split=["train[:80%]", "train[80%:]", "test"],
    as_supervised=True,
    with_info=True
)

print("Dataset loaded successfully.")
print("Number of classes:", ds_info.features["label"].num_classes)
print("Training samples:", tf.data.experimental.cardinality(train_raw).numpy())
print("Validation samples:", tf.data.experimental.cardinality(val_raw).numpy())
print("Test samples:", tf.data.experimental.cardinality(test_raw).numpy())
print("First five classes:", ds_info.features["label"].names[:5])

In [ ]:
def preprocess_image(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

train_data = train_raw.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_data = val_raw.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_data = test_raw.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

train_data = train_data.shuffle(1024, seed=SEED, reshuffle_each_iteration=True).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_data = val_data.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_data = test_data.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

for images, labels in train_data.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image dtype:", images.dtype)

## 1. Baseline MobileNetV2

In [ ]:
def build_classifier(base_trainable=False, dropout_rate=0.0, use_bn=False, l2_rate=0.0, learning_rate=5e-4):
    base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = base_trainable
    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    if use_bn:
        x = layers.BatchNormalization()(x)
    x = layers.Dense(160, activation="relu", kernel_regularizer=(regularizers.l2(l2_rate) if l2_rate else None))(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

baseline_model = build_classifier()
start = time.time()
baseline_history = baseline_model.fit(train_data, validation_data=val_data, epochs=EPOCHS, verbose=1)
baseline_time = time.time() - start
baseline_val_loss, baseline_val_acc = baseline_model.evaluate(val_data, verbose=0)
print(f"Baseline validation accuracy: {baseline_val_acc*100:.2f}%")
print(f"Baseline validation loss: {baseline_val_loss:.4f}")
print(f"Baseline training time: {baseline_time:.2f} s")

## 2. Weight Initialization

In [ ]:
def build_initialization_model(initializer):
    base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = False
    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = layers.GlobalAveragePooling2D()(base(inputs, training=False))
    x = layers.Dense(160, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", kernel_initializer=initializer)(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

initializers_dict = {
    "Zero": keras.initializers.Zeros(),
    "RandomNormal": keras.initializers.RandomNormal(stddev=0.05),
    "GlorotUniform": keras.initializers.GlorotUniform(),
    "HeNormal": keras.initializers.HeNormal(),
}
initialization_histories, initialization_times = {}, {}
for name, init in initializers_dict.items():
    print("\n---", name, "---")
    model = build_initialization_model(init)
    start = time.time()
    hist = model.fit(train_data, validation_data=val_data, epochs=EPOCHS, verbose=1)
    initialization_histories[name] = hist
    initialization_times[name] = time.time() - start
    del model
    keras.backend.clear_session()
    gc.collect()

rows=[]
for name,h in initialization_histories.items():
    rows.append({"Initialization":name,"Final Training Loss":h.history["loss"][-1],"Best Validation Accuracy (%)":max(h.history["val_accuracy"])*100,"Training Time (s)":initialization_times[name]})
initialization_results_df=pd.DataFrame(rows)
initialization_results_df

In [ ]:
plt.figure(figsize=(10,6))
for name,h in initialization_histories.items(): plt.plot(h.history["loss"], marker="o", label=name)
plt.xlabel("Epoch"); plt.ylabel("Training Loss"); plt.title("Plot 1 – Training Loss vs Epoch: Weight Initialization"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"plot_1_initialization_loss.png"),dpi=300); plt.show()

plt.figure(figsize=(10,6))
for name,h in initialization_histories.items(): plt.plot(np.array(h.history["val_accuracy"])*100, marker="o", label=name)
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy (%)"); plt.title("Plot 2 – Validation Accuracy vs Epoch: Weight Initialization"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"plot_2_initialization_accuracy.png"),dpi=300); plt.show()

best_initialization = initialization_results_df.loc[initialization_results_df["Best Validation Accuracy (%)"].idxmax(),"Initialization"]
print("Best initialization by runtime validation accuracy:", best_initialization)

## 3. Regularization and Overfitting

In [ ]:
regularization_configs={
    "No Regularization":dict(),
    "L2 Regularization":dict(l2_rate=5e-4),
    "Dropout 0.30":dict(dropout_rate=0.30),
    "Batch Normalization":dict(use_bn=True),
}
regularization_histories, regularization_times = {}, {}
for name,kwargs in regularization_configs.items():
    print("\n---",name,"---")
    model=build_classifier(learning_rate=5e-4,**kwargs)
    start=time.time(); h=model.fit(train_data,validation_data=val_data,epochs=EPOCHS,verbose=1)
    regularization_histories[name]=h; regularization_times[name]=time.time()-start
    del model; keras.backend.clear_session(); gc.collect()
regularization_results_df=pd.DataFrame([{
    "Regularization":n,"Final Training Loss":h.history["loss"][-1],"Best Validation Accuracy (%)":max(h.history["val_accuracy"])*100,"Training Time (s)":regularization_times[n]
} for n,h in regularization_histories.items()])
regularization_results_df

In [ ]:
plt.figure(figsize=(10,6))
for name,h in regularization_histories.items():
    plt.plot(np.array(h.history["accuracy"])*100,label=f"{name} – Train")
    plt.plot(np.array(h.history["val_accuracy"])*100,linestyle="--",label=f"{name} – Validation")
plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)"); plt.title("Plot 3 – Training and Validation Accuracy: Regularization"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"plot_3_regularization_accuracy.png"),dpi=300); plt.show()

plt.figure(figsize=(10,6))
for name,h in regularization_histories.items():
    plt.plot(h.history["loss"],label=f"{name} – Train")
    plt.plot(h.history["val_loss"],linestyle="--",label=f"{name} – Validation")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Plot 4 – Training and Validation Loss: Regularization"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"plot_4_regularization_loss.png"),dpi=300); plt.show()
best_regularization=regularization_results_df.loc[regularization_results_df["Best Validation Accuracy (%)"].idxmax(),"Regularization"]
print("Best regularization by runtime validation accuracy:",best_regularization)

## 4. Batch Normalization

In [ ]:
bn_results=[]
for name,use_bn in [("Without BN",False),("With BN",True)]:
    model=build_classifier(use_bn=use_bn,learning_rate=5e-4)
    h=model.fit(train_data,validation_data=val_data,epochs=EPOCHS,verbose=1)
    bn_results.append({"Configuration":name,"Best Validation Accuracy (%)":max(h.history["val_accuracy"])*100})
    del model; keras.backend.clear_session(); gc.collect()
bn_results_df=pd.DataFrame(bn_results)
bn_results_df

## 5. Optimization Algorithms

In [ ]:
def build_optimizer_model(name):
    model=build_classifier(learning_rate=5e-4)
    if name=="SGD": opt=keras.optimizers.SGD(learning_rate=5e-4)
    elif name=="Momentum": opt=keras.optimizers.SGD(learning_rate=5e-4,momentum=0.9)
    elif name=="RMSProp": opt=keras.optimizers.RMSprop(learning_rate=5e-4)
    elif name=="Adam": opt=keras.optimizers.Adam(learning_rate=5e-4)
    else: raise ValueError("Unknown optimizer")
    model.compile(optimizer=opt,loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model

optimizer_names=["SGD","Momentum","RMSProp","Adam"]
optimizer_histories,optimizer_times={},{}
for name in optimizer_names:
    print("\n---",name,"---")
    model=build_optimizer_model(name); start=time.time(); h=model.fit(train_data,validation_data=val_data,epochs=EPOCHS,verbose=1)
    optimizer_histories[name]=h; optimizer_times[name]=time.time()-start
    del model; keras.backend.clear_session(); gc.collect()
optimizer_results_df=pd.DataFrame([{
    "Optimizer":n,"Final Loss":h.history["loss"][-1],"Best Validation Accuracy (%)":max(h.history["val_accuracy"])*100,"Epoch of Best Validation Accuracy":int(np.argmax(h.history["val_accuracy"])+1),"Training Time (s)":optimizer_times[n]
} for n,h in optimizer_histories.items()])
optimizer_results_df

In [ ]:
plt.figure(figsize=(10,6))
for n,h in optimizer_histories.items(): plt.plot(h.history["loss"],marker="o",label=n)
plt.xlabel("Epoch"); plt.ylabel("Training Loss"); plt.title("Plot 6 – Training Loss: Optimizers"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"plot_6_optimizer_loss.png"),dpi=300); plt.show()

plt.figure(figsize=(10,6))
for n,h in optimizer_histories.items(): plt.plot(np.array(h.history["val_accuracy"])*100,marker="o",label=n)
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy (%)"); plt.title("Plot 7 – Validation Accuracy: Optimizers"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"plot_7_optimizer_accuracy.png"),dpi=300); plt.show()
best_optimizer=optimizer_results_df.loc[optimizer_results_df["Best Validation Accuracy (%)"].idxmax(),"Optimizer"]
print("Best optimizer by runtime validation accuracy:",best_optimizer)

## 6. CNN Hyperparameter Tuning

In [ ]:
def run_head_experiment(param_name,param_value):
    kwargs={"learning_rate":5e-4}
    if param_name=="Learning Rate": kwargs["learning_rate"]=param_value
    if param_name=="Dropout Rate": kwargs["dropout_rate"]=param_value
    model=build_classifier(**kwargs)
    start=time.time(); h=model.fit(train_data,validation_data=val_data,epochs=EPOCHS,verbose=0)
    result={param_name:param_value,"Validation Accuracy (%)":max(h.history["val_accuracy"])*100,"Training Time (s)":time.time()-start}
    del model; keras.backend.clear_session(); gc.collect(); return result

learning_rate_values=[5e-4,1e-4]
batch_size_values=[16,24,48]
dropout_values=[0.0,0.30,0.50]

learning_rate_results=pd.DataFrame([run_head_experiment("Learning Rate",v) for v in learning_rate_values])
print("Learning-rate results"); display(learning_rate_results)

batch_results=[]
for bs in batch_size_values:
    model=build_classifier(learning_rate=5e-4); start=time.time()
    h=model.fit(train_data.unbatch().batch(bs).prefetch(tf.data.AUTOTUNE),validation_data=val_data,epochs=EPOCHS,verbose=0)
    batch_results.append({"Batch Size":bs,"Validation Accuracy (%)":max(h.history["val_accuracy"])*100,"Training Time (s)":time.time()-start})
    del model; keras.backend.clear_session(); gc.collect()
batch_size_results=pd.DataFrame(batch_results); print("Batch-size results"); display(batch_size_results)

dropout_results=pd.DataFrame([run_head_experiment("Dropout Rate",v) for v in dropout_values])
print("Dropout results"); display(dropout_results)

In [ ]:
fig_specs=[(learning_rate_results,"Learning Rate","Plot 8 – Learning Rate vs Validation Accuracy","plot_8_learning_rate.png"),(batch_size_results,"Batch Size","Plot 9 – Batch Size vs Validation Accuracy","plot_9_batch_size.png"),(dropout_results,"Dropout Rate","Plot 10 – Dropout Rate vs Validation Accuracy","plot_10_dropout.png")]
for df,x,title,fn in fig_specs:
    plt.figure(figsize=(8,6)); plt.plot(df[x],df["Validation Accuracy (%)"],marker="o"); plt.xlabel(x); plt.ylabel("Validation Accuracy (%)"); plt.title(title); plt.grid(True); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,fn),dpi=300); plt.show()

best_learning_rate=float(learning_rate_results.loc[learning_rate_results["Validation Accuracy (%)"].idxmax(),"Learning Rate"])
best_batch_size=int(batch_size_results.loc[batch_size_results["Validation Accuracy (%)"].idxmax(),"Batch Size"])
best_dropout=float(dropout_results.loc[dropout_results["Validation Accuracy (%)"].idxmax(),"Dropout Rate"])
print("Best LR:",best_learning_rate,"Best batch size:",best_batch_size,"Best dropout:",best_dropout)

## 7. Transfer Learning and Fine-Tuning

In [ ]:
def build_transfer_model(fine_tune_layers=0, learning_rate=5e-4):
    base=MobileNetV2(input_shape=(*IMG_SIZE,3),include_top=False,weights="imagenet")
    base.trainable=fine_tune_layers>0
    if fine_tune_layers>0:
        for layer in base.layers[:-fine_tune_layers]: layer.trainable=False
        for layer in base.layers:
            if isinstance(layer,layers.BatchNormalization): layer.trainable=False
    inputs=keras.Input(shape=(*IMG_SIZE,3)); x=layers.GlobalAveragePooling2D()(base(inputs,training=False)); x=layers.Dense(160,activation="relu")(x); x=layers.Dropout(0.30)(x); outputs=layers.Dense(NUM_CLASSES,activation="softmax")(x)
    model=keras.Model(inputs,outputs); model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),loss="sparse_categorical_crossentropy",metrics=["accuracy"]); return model

feature_model=build_transfer_model(); start=time.time(); feature_history=feature_model.fit(train_data,validation_data=val_data,epochs=EPOCHS,verbose=1); feature_time=time.time()-start
keras.backend.clear_session(); gc.collect()

fine_model=build_transfer_model(fine_tune_layers=30,learning_rate=1e-5); trainable_count=sum(int(l.trainable) for l in fine_model.layers); print("Trainable model layers:",trainable_count); start=time.time(); fine_history=fine_model.fit(train_data,validation_data=val_data,epochs=EPOCHS,verbose=1); fine_time=time.time()-start

transfer_learning_results=pd.DataFrame({"Configuration":["Feature Extraction","Fine-Tuning (last 30 layers)"],"Best Validation Accuracy (%)":[max(feature_history.history["val_accuracy"])*100,max(fine_history.history["val_accuracy"])*100],"Final Validation Loss":[feature_history.history["val_loss"][-1],fine_history.history["val_loss"][-1]],"Training Time (s)":[feature_time,fine_time],"Parameters":[feature_parameter_count,fine_model.count_params()]})
transfer_learning_results

In [ ]:
plt.figure(figsize=(10,6)); plt.plot(np.array(feature_history.history["val_accuracy"])*100,marker="o",label="Feature Extraction"); plt.plot(np.array(fine_history.history["val_accuracy"])*100,marker="o",label="Fine-Tuning"); plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy (%)"); plt.title("Plot 11 – Feature Extraction vs Fine-Tuning"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"plot_11_transfer_accuracy.png"),dpi=300); plt.show()

plt.figure(figsize=(10,6)); plt.plot(feature_history.history["loss"],marker="o",label="Feature – Train"); plt.plot(feature_history.history["val_loss"],marker="o",linestyle="--",label="Feature – Val"); plt.plot(fine_history.history["loss"],marker="o",label="Fine-Tune – Train"); plt.plot(fine_history.history["val_loss"],marker="o",linestyle="--",label="Fine-Tune – Val"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Plot 12 – Transfer Learning Loss Curves"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"plot_12_transfer_loss.png"),dpi=300); plt.show()

## Runtime summary
The following summary is generated only from the actual execution of this notebook. This avoids hard-coded or fabricated performance values.

In [ ]:
summary=pd.DataFrame({"Item":["Baseline","Best Initialization","Best Regularization","Best Optimizer","Best Learning Rate","Best Batch Size","Best Dropout","Fine-Tuning Layers"],"Value":[f"{baseline_val_acc*100:.2f}%",best_initialization,best_regularization,best_optimizer,best_learning_rate,best_batch_size,best_dropout,30]})
summary.to_csv(os.path.join(OUTPUT_DIR,"part1_summary.csv"),index=False)
summary